## 1. Configuration

In [2]:
!pip install lifelines

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.3 MB 599.9 kB/s eta 0:00:19
   - -------------------------------------- 0.5/11.3 MB 599.9 kB/s eta 0:00:19
   - -------------------------------------- 0.5/11.3 MB 599.9 kB/s eta 0:00:19
   - -------------------------------------- 0.5/11.3 MB 599.9 kB/s eta 0:00:19
   -

  You can safely remove it manually.


In [3]:
import os, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from lifelines import CoxPHFitter
warnings.filterwarnings("ignore")
SEED=42; np.random.seed(SEED)

FEATURES_CSV = r"C:\Users\FatmaH\dataset\Step2_features_final\tcga_features_with_clinical.csv"
OUT_DIR      = r"C:\Users\FatmaH\dataset\Stage3a_mri_risk"; os.makedirs(OUT_DIR,exist_ok=True)

ID, TIME, EVENT, SUBTYPE, SOURCE = "tcga_id","os_days","os_event","subtype","mask_source"
SUBTYPES = ["Astrocytoma","Oligodendroglioma","Glioblastoma"]
PUBLISHED = {"Astrocytoma":0.655,"Oligodendroglioma":0.688,"Glioblastoma":0.588}

PENALIZER, L1_RATIO = 0.5, 0.5      # Elastic-Net Cox, matches the lab convention
OUTER_FOLDS, N_REPEATS = 5, 5
EVENTS_PER_VAR = 10                 # screening budget

print("features file:", "ok" if os.path.isfile(FEATURES_CSV) else "MISSING")

features file: ok


## 2. Load

In [5]:
df = pd.read_csv(FEATURES_CSV)

DROP = {ID,TIME,EVENT,SUBTYPE,SOURCE,"brats_folder","collection","mgmt_mapping",
        "residual_vol","status","folder","multifocal","n_components",
        "ncr_et_ratio"}          # keep the log version instead
feat_cols = [c for c in df.columns
             if c not in DROP and pd.api.types.is_numeric_dtype(df[c])]
# low_enhancement is a flag, keep it as a covariate
print(f"patients: {len(df)}   features ({len(feat_cols)}): {feat_cols}\n")

df = df[df[TIME].notna() & (df[TIME]>0) & df[EVENT].isin([0,1])].reset_index(drop=True)
df["subtype_idx"] = df[SUBTYPE].map({s:i for i,s in enumerate(SUBTYPES)})

print(df.groupby(SUBTYPE).agg(n=(EVENT,"size"),events=(EVENT,"sum"),
      median_OS=(TIME,"median")).reindex(SUBTYPES).to_string())
print(f"\nPOOLED n={len(df)}, events={int(df[EVENT].sum())}")
print("\nby mask source:")
print(df.groupby(SOURCE).agg(n=(EVENT,"size"),events=(EVENT,"sum")).to_string())
print(f"\nNaNs in features: {int(df[feat_cols].isna().sum().sum())}")

patients: 213   features (14): ['et_volume', 'ncr_volume', 'ed_volume', 'wt_volume', 'tc_volume', 'ce_rim_width', 'surface_regularity', 'max_diameter', 'solidity', 'sphericity', 'et_wt_ratio', 'ncr_et_ratio_log', 'centroid_dev', 'low_enhancement']

                     n  events  median_OS
subtype                                  
Astrocytoma         61      11    517.480
Oligodendroglioma   27       5    432.248
Glioblastoma       125      93    340.928

POOLED n=213, events=109

by mask source:
               n  events
mask_source             
expert       143      70
model         70      39

NaNs in features: 37


## 3. Metrics and model

In [6]:
def c_index(t,e,r):
    """Harrell's C, risk orientation. Verified identical to lifelines.concordance_index(t,-r,e)."""
    t=np.asarray(t,float); e=np.asarray(e,int); r=np.asarray(r,float)
    num=den=0.0
    for i in range(len(t)):
        if e[i]!=1: continue
        L=t>t[i]; den+=L.sum(); num+=(r[i]>r[L]).sum()+0.5*(r[i]==r[L]).sum()
    return num/den if den>0 else np.nan

def c_within(t,e,r,s):
    """Concordance within each subtype, averaged. A score that only encodes
    subtype baseline prognosis scores ~0.5 here, which is the point."""
    cs=[]
    for u in np.unique(s):
        m=s==u
        if m.sum()>4 and e[m].sum()>=2:
            v=c_index(t[m],e[m],r[m])
            if np.isfinite(v): cs.append(v)
    return float(np.mean(cs)) if cs else np.nan


def screen(X, t, e, k):
    """Rank features by univariate Cox |z|, keep top-k. TRAIN FOLD ONLY."""
    sc, keep = [], []
    for j in range(X.shape[1]):
        x = X[:,j]
        if np.var(x) < 1e-8: continue
        try:
            f = CoxPHFitter(penalizer=0.1).fit(pd.DataFrame({"x":x,"T":t,"E":e}), "T","E")
            sc.append(abs(f.params_["x"]/max(f.standard_errors_["x"],1e-8))); keep.append(j)
        except Exception:
            continue
    if not keep: return np.arange(min(k, X.shape[1]))
    return np.array([keep[i] for i in np.argsort(sc)[::-1][:k]])


def run_cv(sub, cols, stratify_by_subtype, label):
    """Repeated CV. Returns (fold C-index list, out-of-fold risk score, coefficient list)."""
    X = sub[cols].astype(float).fillna(0).values
    t = sub[TIME].astype(float).values
    e = sub[EVENT].astype(int).values
    s = sub["subtype_idx"].astype(int).values
    n = len(t)

    oof = np.zeros(n); cnt = np.zeros(n); fold_c = []; coefs = []
    strat_lbl = (s*2+e) if stratify_by_subtype else e

    for rep in range(N_REPEATS):
        skf = StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=SEED+rep)
        for tr, te in skf.split(np.zeros(n), strat_lbl):
            if e[te].sum() < 2 or e[tr].sum() < 5: continue

            k = max(3, int(e[tr].sum()) // EVENTS_PER_VAR)      # event budget
            idx = screen(X[tr], t[tr], e[tr], k)

            mu, sd = X[tr][:,idx].mean(0), X[tr][:,idx].std(0)+1e-8
            names = [cols[j] for j in idx]
            d = pd.DataFrame((X[:,idx]-mu)/sd, columns=names)
            d["T"]=t; d["E"]=e
            if stratify_by_subtype: d["S"]=s
            try:
                f = CoxPHFitter(penalizer=PENALIZER, l1_ratio=L1_RATIO).fit(
                        d.iloc[tr], "T","E", strata=["S"] if stratify_by_subtype else None)
                r = np.log(f.predict_partial_hazard(d).values + 1e-12)
                coefs.append(f.params_)
            except Exception:
                continue

            score = (c_within(t[te],e[te],r[te],s[te]) if stratify_by_subtype
                     else c_index(t[te],e[te],r[te]))
            if np.isfinite(score): fold_c.append(score)
            oof[te] += r[te]; cnt[te] += 1

    oof = oof/np.maximum(cnt,1)
    oof_c = (c_within(t,e,oof,s) if stratify_by_subtype else c_index(t,e,oof))
    print(f"{label}")
    print(f"   n={n}  events={int(e.sum())}  folds={len(fold_c)}  features/fold~{k}")
    print(f"   CV C-index      : {np.nanmean(fold_c):.4f} +/- {np.nanstd(fold_c):.4f}")
    print(f"   OOF score       : {oof_c:.4f}")
    return fold_c, oof, coefs

## 4. Primary — Glioblastoma only

93 events. Single subtype, so no stratification needed and plain concordance is the right metric.

In [7]:
gbm = df[df[SUBTYPE]=="Glioblastoma"].reset_index(drop=True)
print("="*60)
fc_gbm, oof_gbm, coef_gbm = run_cv(gbm, feat_cols, False, "GBM (primary)")
print("="*60)
print(f"   published genomic benchmark (GBM): {PUBLISHED['Glioblastoma']:.3f}")
d = np.nanmean(fc_gbm) - PUBLISHED['Glioblastoma']
print(f"   MRI-only vs that benchmark      : {d:+.4f}")
print("\n   Note: this is MRI alone vs a genomics-only model - not a fair head-to-head,")
print("   but it indicates whether imaging carries comparable prognostic signal.")

gbm_out = gbm[[ID,SUBTYPE,SOURCE,TIME,EVENT]].copy()
gbm_out["mri_risk_score"] = oof_gbm
gbm_out.to_csv(os.path.join(OUT_DIR,"mri_risk_score_gbm.csv"), index=False)

GBM (primary)
   n=125  events=93  folds=25  features/fold~7
   CV C-index      : 0.6074 +/- 0.0703
   OOF score       : 0.5811
   published genomic benchmark (GBM): 0.588
   MRI-only vs that benchmark      : +0.0194

   Note: this is MRI alone vs a genomics-only model - not a fair head-to-head,
   but it indicates whether imaging carries comparable prognostic signal.


## 5. Secondary — pooled, subtype-stratified

All 213 patients share one coefficient vector while each subtype keeps its own baseline hazard.
Scored **within subtype**, so separating GBM from Oligodendroglioma earns nothing.

In [8]:
print("="*60)
fc_pool, oof_pool, coef_pool = run_cv(df, feat_cols, True, "POOLED subtype-stratified (secondary)")
print("="*60)

pool_out = df[[ID,SUBTYPE,SOURCE,TIME,EVENT]].copy()
pool_out["mri_risk_score"] = oof_pool
pool_out.to_csv(os.path.join(OUT_DIR,"mri_risk_score_pooled.csv"), index=False)

# per-subtype breakdown of the pooled score
t=df[TIME].values; e=df[EVENT].values; s=df["subtype_idx"].values
print("\nper-subtype C-index of the pooled OOF score:")
for i,st in enumerate(SUBTYPES):
    m=s==i
    if m.sum()>4 and e[m].sum()>=2:
        print(f"   {st:20} n={int(m.sum()):3d} events={int(e[m].sum()):3d} "
              f"C={c_index(t[m],e[m],oof_pool[m]):.4f}   (published {PUBLISHED[st]:.3f})")
    else:
        print(f"   {st:20} n={int(m.sum()):3d} events={int(e[m].sum()):3d} - too few to score")

POOLED subtype-stratified (secondary)
   n=213  events=109  folds=25  features/fold~8
   CV C-index      : 0.6141 +/- 0.1916
   OOF score       : 0.6394

per-subtype C-index of the pooled OOF score:
   Astrocytoma          n= 61 events= 11 C=0.6889   (published 0.655)
   Oligodendroglioma    n= 27 events=  5 C=0.6000   (published 0.688)
   Glioblastoma         n=125 events= 93 C=0.6292   (published 0.588)


## 6. Sensitivity — expert masks only

Repeats both analyses on the 143 expert-annotated patients. Similar numbers mean the mixed mask
source is not driving anything.

In [9]:
exp_df = df[df[SOURCE]=="expert"].reset_index(drop=True)
exp_gbm = exp_df[exp_df[SUBTYPE]=="Glioblastoma"].reset_index(drop=True)

print("="*60)
res={}
if int(exp_gbm[EVENT].sum())>=25:
    fc,_ ,_ = run_cv(exp_gbm, feat_cols, False, "GBM, expert masks only")
    res["gbm_expert"]=np.nanmean(fc)
else:
    print(f"GBM expert-only: {int(exp_gbm[EVENT].sum())} events - too few")
print()
if int(exp_df[EVENT].sum())>=25:
    fc,_,_ = run_cv(exp_df, feat_cols, True, "POOLED, expert masks only")
    res["pooled_expert"]=np.nanmean(fc)
print("="*60)

print("\nfull cohort vs expert-only:")
print(f"   GBM    : {np.nanmean(fc_gbm):.4f}  vs  {res.get('gbm_expert',float('nan')):.4f}")
print(f"   pooled : {np.nanmean(fc_pool):.4f}  vs  {res.get('pooled_expert',float('nan')):.4f}")
print("\nDifferences under ~0.03 indicate the mask source is not driving the result;")
print("state that explicitly in the write-up.")

GBM, expert masks only
   n=87  events=62  folds=25  features/fold~5
   CV C-index      : 0.6080 +/- 0.1015
   OOF score       : 0.5477

POOLED, expert masks only
   n=143  events=70  folds=25  features/fold~5
   CV C-index      : 0.5662 +/- 0.1247
   OOF score       : 0.6831

full cohort vs expert-only:
   GBM    : 0.6074  vs  0.6080
   pooled : 0.6141  vs  0.5662

Differences under ~0.03 indicate the mask source is not driving the result;
state that explicitly in the write-up.


## 7. Which imaging features carry prognostic signal

Mean |coefficient| across folds. Elastic-Net drives uninformative features toward zero, so the
survivors are your candidate imaging markers.

In [10]:
def show_coefs(coefs, title):
    if not coefs:
        print(f"{title}: none"); return None
    cf = pd.DataFrame(coefs)
    cf.columns = [c[1] if isinstance(c,tuple) else c for c in cf.columns]
    ma = cf.abs().mean().sort_values(ascending=False)
    sel = (cf.abs() > 0.01).mean()          # how often retained
    print(f"\n{title}")
    print(f"{'feature':24}{'mean|coef|':>12}{'selected%':>11}")
    for k,v in ma.items():
        print(f"{str(k):24}{v:12.4f}{100*sel.get(k,0):10.0f}%  {'#'*int(v*30)}")
    return ma

ma_gbm = show_coefs(coef_gbm, "GBM (primary)")
ma_pool = show_coefs(coef_pool, "POOLED subtype-stratified")
if ma_gbm is not None:
    ma_gbm.to_csv(os.path.join(OUT_DIR,"mri_feature_importance_gbm.csv"))


GBM (primary)
feature                   mean|coef|  selected%
sphericity                    0.0353        72%  #
solidity                      0.0061        32%  
max_diameter                  0.0004         4%  
ed_volume                     0.0000         0%  
wt_volume                     0.0000         0%  
centroid_dev                  0.0000         0%  
et_volume                     0.0000         0%  
ce_rim_width                  0.0000         0%  
tc_volume                     0.0000         0%  

POOLED subtype-stratified
feature                   mean|coef|  selected%
sphericity                    0.0000         0%  
solidity                      0.0000         0%  
et_volume                     0.0000         0%  
ce_rim_width                  0.0000         0%  
surface_regularity            0.0000         0%  
low_enhancement               0.0000         0%  
ncr_et_ratio_log              0.0000         0%  
et_wt_ratio                   0.0000         0%  
ncr_volume 

## 8. Kaplan-Meier risk stratification (out-of-fold, so honest)

In [ ]:
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

def km(t,e,r,title,ax):
    if len(t)<10 or e.sum()<5:
        ax.set_title(f"{title}\n(insufficient events)"); ax.axis("off"); return
    hi = r > np.median(r)
    kmf = KaplanMeierFitter()
    for m,lab,col in [(~hi,"Low risk","#1C7293"),(hi,"High risk","#C0392B")]:
        if m.sum()>1:
            kmf.fit(t[m],e[m],label=lab); kmf.plot_survival_function(ax=ax,color=col)
    lr = logrank_test(t[hi],t[~hi],e[hi],e[~hi])
    ax.set_title(f"{title}\nlog-rank p = {lr.p_value:.2e}")
    ax.set_xlabel("Days"); ax.set_ylabel("Survival")

fig,axes = plt.subplots(1,2,figsize=(11,4.4))
km(gbm[TIME].values, gbm[EVENT].values, oof_gbm, "GBM — MRI risk score", axes[0])
km(df[TIME].values,  df[EVENT].values,  oof_pool, "Pooled — MRI risk score", axes[1])
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR,"km_mri_risk.png"),dpi=150); plt.show()
print("Splits use the OUT-OF-FOLD score, so these are honest, not in-sample.")